In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset
file_path = "/Users/benvarvill/Documents/APS Contacted Uncontacted.xlsx"
df = pd.read_excel(file_path)

# Define categorical and numerical columns
categorical_cols = ['Type', 'Source', 'Contacted?', 'Tracker Theme', 'Country']
numerical_cols = []  # No numerical columns were detected apart from 'Result'

# Convert categorical columns to string type
df[categorical_cols] = df[categorical_cols].astype(str)

# Target Encode 'Country' (High Cardinality)
country_means = df.groupby('Country')['Result'].mean()
df['Country_encoded'] = df['Country'].map(country_means)

# One-Hot Encode other categorical variables
encoder = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
X_categorical = encoder.fit_transform(df[['Type', 'Source', 'Contacted?', 'Tracker Theme']])

# Combine processed categorical data
X_processed = np.hstack((X_categorical, df[['Country_encoded']].values))

# Define target variable
y = df['Result'].astype(int)

# Train-Test Split (80% training, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

# Train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf_model.predict(X_test)

# Evaluate performance
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Model Accuracy: {accuracy_rf:.2f}")
print(classification_report(y_test, y_pred_rf))

Random Forest Model Accuracy: 0.80
              precision    recall  f1-score   support

           0       0.92      0.86      0.89       314
           1       0.13      0.23      0.17        30

    accuracy                           0.80       344
   macro avg       0.53      0.55      0.53       344
weighted avg       0.85      0.80      0.83       344

